# Chapter 8 — Combining channels and recognising triples

**Goal.** Join solutions without duplicating stars, compare independent orbital periods, and understand what
additional measurements can—and cannot—resolve. This chapter uses only the existing `nss_two_body` release.

One orbit can produce an astrometric wobble, radial-velocity variation and eclipses. A hierarchical system
can instead produce a short-period spectroscopic/eclipsing inner binary and a long-period astrometric outer
orbit. Misidentification, aliases and false solutions can also produce disagreeing periods. Agreement is a
check; disagreement is a question to investigate, not an automatic triple classification.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.3/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 8.1 Count solutions before joining stars

The NSS table has one row per solution, not necessarily one row per source. `AstroSpectroSB1` and
`EclipsingSpectro` are combined fits; their shared parameters are not independent confirmations from two
rows. Separate `Orbital` and `SB1` rows, in contrast, allow a comparison of separately fitted periods,
although they still share the Gaia mission and its systematics. The published combined-solution counts in
Arenou et al. (2023), https://arxiv.org/abs/2206.05595, provide a catalogue check.

In [ ]:
nss = load('nss_two_body')
counts = nss.nss_solution_type.value_counts()
fig, ax = plt.subplots(figsize=(8, 3.5))
labels = ['Orbital', 'SB1', 'EclipsingBinary', 'AstroSpectroSB1', 'EclipsingSpectro']
ax.bar(labels, counts.reindex(labels), color=['C0', 'C1', 'C2', 'C3', 'C4'])
ax.set(yscale='log', ylabel='solution rows'); ax.tick_params(axis='x', labelrotation=15)
plt.tight_layout(); plt.show()
assert counts['AstroSpectroSB1'] == 33467 and counts['EclipsingSpectro'] == 155  # published exact counts
assert len(nss) > nss.source_id.nunique()     # confirms why an unrestricted self-join is unsafe

**Figure 8.1.** Counts of five explicitly selected NSS solution types, coloured by type; the vertical
axis is logarithmic. Counts refer to rows, not disjoint sets of stars. The two combined-solution counts
reproduce the published totals. In particular, counting a joint solution twice would invent an additional
measurement rather than add information.

## 8.2 Compare separately fitted periods

Select one solution type on each side, assert uniqueness of its source IDs, and ask pandas for a one-to-one
merge. Keep the ID as a 64-bit integer: floating-point conversion can silently merge or lose distinct Gaia IDs.
Compare `Orbital` with `SB1`, then `Orbital` with `EclipsingBinary`. Plot reference lines for the same period
and a factor of two. Formal error bars can be misleading when a pipeline chose the wrong alias, so the
period-ratio diagram is a diagnostic rather than a universal significance test.

In [ ]:
def pair_solutions(first, second):
    cols = ['source_id', 'period', 'period_error', 'eccentricity', 'goodness_of_fit', 'significance']
    a = nss.loc[nss.nss_solution_type == first, cols]
    b = nss.loc[nss.nss_solution_type == second, cols]
    assert a.source_id.is_unique and b.source_id.is_unique
    assert a.source_id.dtype.kind in 'iu' and b.source_id.dtype.kind in 'iu'
    return a.merge(b, on='source_id', suffixes=('_a', '_b'), validate='one_to_one')

rv_pairs = pair_solutions('Orbital', 'SB1')
eclipse_pairs = pair_solutions('Orbital', 'EclipsingBinary')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, pairs, title in zip(ax, [rv_pairs, eclipse_pairs], ['Separate SB1 orbit', 'Separate eclipsing orbit']):
    a.scatter(pairs.period_a, pairs.period_b, s=9, alpha=0.4)
    x = np.geomspace(min(pairs.period_a.min(), pairs.period_b.min()),
                     max(pairs.period_a.max(), pairs.period_b.max()), 100)
    a.plot(x, x, 'k-', lw=1, label='equal period')
    a.plot(x, 0.5*x, 'k:', lw=1, label='factor of two')
    a.plot(x, 2*x, 'k:', lw=1)
    a.set(xscale='log', yscale='log', xlabel='astrometric period [days]',
          ylabel='other-channel period [days]', title=f'{title}: {len(pairs):,} sources')
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()
# Join integrity: exact set intersections, not an assumed astrophysical agreement fraction.
for pairs, second in [(rv_pairs, 'SB1'), (eclipse_pairs, 'EclipsingBinary')]:
    expected_ids = set(nss.loc[nss.nss_solution_type == 'Orbital', 'source_id']) & set(nss.loc[nss.nss_solution_type == second, 'source_id'])
    assert set(pairs.source_id) == expected_ids and pairs.source_id.is_unique

**Figure 8.2.** Periods of exact-ID matched, separately fitted solutions: astrometry versus SB1 (left)
and versus eclipses (right), in days on logarithmic axes. Blue points each represent one source; the solid
black line is equality, with dotted factor-of-two lines. The SB1 comparison includes both agreement and
large discrepancies. The eclipse comparison contains strongly separated time scales. These are candidate
interpretations to test with epoch data, imaging and independent velocities; the plot does not count confirmed
triples or measure their occurrence rate.

## 8.3 What eclipses add to a double-lined orbit

For a double-lined binary, $q=M_2/M_1=K_1/K_2$. With $P$ in seconds and velocities in m/s,

$$M_1\sin^3i=\frac{P(1-e^2)^{3/2}(K_1+K_2)^2K_2}{2\pi G},\qquad
M_2\sin^3i=qM_1\sin^3i.$$

Spectroscopy provides projected masses. An eclipse model can constrain inclination, provided its radius,
brightness, dilution and limb-darkening assumptions are adequate. Demonstrate this with a synthetic binary
of known masses; no catalogue mass is inferred from an unvalidated light curve here.

In [ ]:
G, MSUN, AU, DAY = 6.67430e-11, 1.98847e30, 149597870700.0, 86400.0
M1, M2, Pday, e, inc = 1.2, 0.8, 5.0, 0.2, np.deg2rad(87)
# Kepler's law sets the relative orbit; the two barycentric radii divide it in inverse mass ratio.
a = (G*(M1+M2)*MSUN*(Pday*DAY/(2*np.pi))**2)**(1/3)
K1 = 2*np.pi*a*M2/(M1+M2)*np.sin(inc)/(Pday*DAY*np.sqrt(1-e*e))
K2 = K1*M1/M2
projected1 = Pday*DAY*(1-e*e)**1.5*(K1+K2)**2*K2/(2*np.pi*G*MSUN)
projected2 = projected1*K1/K2
inclinations = np.linspace(20, 90, 300)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(inclinations, projected1/np.sin(np.deg2rad(inclinations))**3, label='primary mass')
ax.plot(inclinations, projected2/np.sin(np.deg2rad(inclinations))**3, label='secondary mass')
ax.axvline(87, color='k', ls='--', label='injected inclination')
ax.scatter([87, 87], [M1, M2], color='k', zorder=5)
ax.set(xlabel='assumed inclination [degrees]', ylabel='inferred mass [solar masses]', yscale='log')
ax.legend(); plt.tight_layout(); plt.show()
# Recover independently injected masses from their synthetic velocities to floating-point precision.
assert np.allclose([projected1, projected2]/np.sin(inc)**3, [M1, M2], rtol=1e-12, atol=0)
assert abs(K1/K2-M2/M1) < 1e-12

**Figure 8.3.** Masses inferred from the same synthetic SB2 velocities as a function of assumed inclination
in degrees: primary (blue) and secondary (orange), with mass in solar units on a logarithmic axis. The dashed
line and black points mark the known injected inclination and masses. The two masses are recovered at that
inclination. A photometric inclination can remove the spectroscopic ambiguity, but only if both channels
measure the same orbit and the photometric model is reliable.

## 8.4 Two time scales in a hierarchical system

Take an equal-mass, equal-light inner pair and a third star. At leading hierarchical order, write the inner
stars' positions as the inner barycentre's wide-orbit position plus opposite short-period displacements.
Their unresolved equal-light photocentre cancels the inner motion, but their separate spectral lines retain
it. This is a kinematic demonstration; it neglects three-body perturbations and is not an N-body stability test.
We use an inner 4-day orbit and outer 600-day orbit, with masses 1, 1 and 0.5 solar masses. Let the outer star
be faint enough to ignore in the light centroid.

In [ ]:
Pin, Pout = 4*DAY, 600*DAY
m1, m2, m3 = 1.0, 1.0, 0.5
ain = (G*(m1+m2)*MSUN*(Pin/(2*np.pi))**2)**(1/3)/AU
aout = (G*(m1+m2+m3)*MSUN*(Pout/(2*np.pi))**2)**(1/3)/AU
outer_radius = aout*m3/(m1+m2+m3)
time = np.linspace(0, 1200, 24001)
outer = outer_radius*np.cos(2*np.pi*time/600)
inner = (ain/2)*np.cos(2*np.pi*time/4)
x1, x2 = outer+inner, outer-inner
photocentre = (x1+x2)/2                    # equal light, so the inner positions cancel
v_outer = -outer_radius*AU*2*np.pi/Pout*np.sin(2*np.pi*time/600)/1000
v_inner = -(ain/2)*AU*2*np.pi/Pin*np.sin(2*np.pi*time/4)/1000
fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
ax[0].plot(time, x1, color='0.7', lw=0.5, label='one inner star')
ax[0].plot(time, photocentre, color='C0', label='unresolved inner-pair photocentre')
ax[0].set(xlabel='time [days]', ylabel='projected position [au]'); ax[0].legend(fontsize=7)
short = time < 20
ax[1].plot(time[short], (v_outer+v_inner)[short], label='star 1')
ax[1].plot(time[short], (v_outer-v_inner)[short], label='star 2')
ax[1].set(xlabel='time [days]', ylabel='line-of-sight velocity [km/s]'); ax[1].legend()
plt.tight_layout(); plt.show()
assert np.max(np.abs(photocentre-outer)) < 1e-12  # exact equal-light cancellation in au
# The velocities of the equal-mass inner pair average to their barycentre's velocity.
assert np.max(np.abs(((v_outer+v_inner)+(v_outer-v_inner))/2-v_outer)) < 1e-12

**Figure 8.4.** A simulated hierarchical system: projected position in au over 1200 days (left), with one
inner star in grey and the inner pair's light centroid in blue; velocities in km/s over the first 20 days
(right), blue and orange for the two inner stars. Both projections were chosen to have full orbital amplitude.
The photocentre follows the 600-day orbit while the separate velocities reveal the 4-day orbit. A real spectrum
with blended lines may not recover these individual velocities cleanly.

For a real candidate, first check IDs, solution types, period aliases and flags. Then test which orbit each
channel measures. Establishing a triple requires additional evidence and a dynamically viable model; fitting
three Keplerian curves by itself is not that evidence. Chapter 9 applies the same discipline to dark companions.

**Exercises.** Give the two inner stars unequal light. How much short-period photocentre motion returns?
List the additional observations needed to distinguish a hierarchical triple from a false astrometric orbit
for one of the period-discrepant sources in Figure 8.2.

## References

- Gaia Collaboration, Arenou et al. (2023), *Gaia Data Release 3: Stellar multiplicity, a teaser for the hidden treasure*, A&A 674, A34: https://arxiv.org/abs/2206.05595 ; https://doi.org/10.1051/0004-6361/202243782